# Dự báo nhiệt độ Hà Nội bằng Random Forest

**Bài toán.** Tại giờ $t$, sử dụng các biến thời tiết đã quan sát để dự báo `temperature_2m` tại $t+1$ giờ, đơn vị °C. Random Forest Regressor kết hợp nhiều cây quyết định được huấn luyện trên các mẫu bootstrap và tập biến ngẫu nhiên; dự báo cuối cùng là trung bình dự báo từ các cây.

Hai cách biểu diễn nhãn được kiểm tra trên cùng dữ liệu:

| Phương án | Nhãn mô hình học | Dự báo nhiệt độ tại $t+1$ |
|---|---|---|
| Trực tiếp | $T_{t+1}$ | Giá trị Random Forest dự báo |
| Theo biến thiên | $T_{t+1}-T_t$ | $T_t$ cộng với biến thiên dự báo |
| Persistence | — | $T_t$ |

**Chạy lại.** Mở notebook trong kernel Python có các thư viện ở `requirements.txt`, sau đó chọn **Run All**. Notebook sử dụng dữ liệu processed và pipeline dùng chung của repo. Bảng, biểu đồ và kết quả được hiển thị trực tiếp trong notebook; mô hình và dự đoán được lưu ở `model/random_forest_outputs/`.

## 1. Cấu hình và hợp đồng dữ liệu

Notebook kế thừa danh sách feature ML đã được nhóm chốt trong `data_processing/feature_selection.json`. Mỗi mẫu là một dòng dữ liệu tại giờ $t$, gồm các biến thời tiết hiện tại, đặc trưng chu kỳ thời gian và thông tin nhiệt độ quá khứ. Không chuẩn hóa dữ liệu vì Random Forest không yêu cầu scaling.

Hai file processed được kiểm tra SHA-256 theo báo cáo feature selection. Chấp nhận cả bản xuống dòng LF và CRLF để tránh lỗi khác biệt khi checkout trên Windows.

In [ ]:
from hashlib import sha256
from io import BytesIO
from pathlib import Path
import json
import sys
from time import perf_counter

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
from IPython.display import Markdown, display
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score, root_mean_squared_error
from sklearn.model_selection import ParameterGrid
import joblib

ROOT = Path.cwd().resolve()
if ROOT.name == "model":
    ROOT = ROOT.parent
if not (ROOT / "src/weather_data.py").is_file():
    raise FileNotFoundError("Hãy chạy notebook từ thư mục gốc dự án hoặc thư mục model.")
sys.path.insert(0, str(ROOT))

from src.weather_data import build_forecast_dataset, load_feature_selection

TRAIN_PATH = ROOT / "data/processed/Hanoi_final.csv"
TEST_PATH = ROOT / "data/processed/Hanoi_test_final.csv"
REPORT_PATH = ROOT / "data_processing/feature_selection.json"
OUTPUT_DIR = ROOT / "model/random_forest_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TARGET = "temperature_2m"
HORIZON = 1
VALIDATION_START = pd.Timestamp("2025-01-01")
TEST_START = pd.Timestamp("2026-01-01")
SEED = 42
N_ESTIMATORS = 150
N_JOBS = -1

report = json.loads(REPORT_PATH.read_text(encoding="utf-8"))
features = load_feature_selection(REPORT_PATH, TARGET, HORIZON)
if not pd.Timestamp(report["selection_train_label_end"]) < VALIDATION_START:
    raise ValueError("Báo cáo chọn biến phải được tạo từ các nhãn trước validation.")

print(f"Bộ biến {report['active_feature_set']} • {len(features)} biến")
print(f"Python {sys.version.split()[0]} | scikit-learn {sklearn.__version__} | "
      f"pandas {pd.__version__} | NumPy {np.__version__} | Matplotlib {matplotlib.__version__}")
print(f"Random Forest: {N_ESTIMATORS} cây/cấu hình | n_jobs={N_JOBS} | random_state={SEED}")
display(Markdown("**Thứ tự biến đầu vào**\n\n" + ", ".join(f"`{feature}`" for feature in features)))

## 2. Đọc dữ liệu, ghép nhãn và chia theo thời gian

`build_forecast_dataset` ghép quan sát tại `time` với nhãn `label_time = time + 1 giờ`, đồng thời kiểm tra chuỗi giờ liên tục và dữ liệu hợp lệ. Ranh giới chia tập dựa trên **giờ của nhãn** theo giờ Hà Nội.

| Tập | Giờ của nhãn | Mục đích |
|---|---|---|
| Train | Trước 01/01/2025 | Huấn luyện các cấu hình Random Forest |
| Validation | Năm 2025 | Chọn cách biểu diễn nhãn và siêu tham số |
| Test | Từ 01/01/2026 đến hết dữ liệu test | Đánh giá cuối cùng sau khi chốt cấu hình |

Không xáo trộn dữ liệu theo kiểu random split. Dữ liệu test chỉ được nạp sau khi đã chọn cấu hình trên Validation.

In [ ]:
def read_processed(path, hash_key):
    content = path.read_bytes()
    expected = report[hash_key]
    if sha256(content).hexdigest() != expected:
        normalized = content.replace(b"\r\n", b"\n")
        if sha256(normalized).hexdigest() != expected:
            raise ValueError(f"{path.name} không khớp SHA-256 trong báo cáo chọn biến.")
    return pd.read_csv(BytesIO(content), usecols=["time", *features], parse_dates=["time"])


def describe_split(frame):
    return {
        "Số mẫu": len(frame),
        "Nhãn đầu": frame["label_time"].min(),
        "Nhãn cuối": frame["label_time"].max(),
    }


def score(actual, predicted):
    return {
        "MAE (°C)": mean_absolute_error(actual, predicted),
        "RMSE (°C)": root_mean_squared_error(actual, predicted),
        "R²": r2_score(actual, predicted),
    }


def predict_temperature(model, inputs, target_mode):
    prediction = model.predict(inputs)
    if target_mode == "Biến thiên":
        prediction = inputs[TARGET].to_numpy() + prediction
    return prediction


development = read_processed(TRAIN_PATH, "processed_sha256")
if development["time"].iloc[-1] != TEST_START - pd.Timedelta(hours=1):
    raise ValueError("CSV huấn luyện phải kết thúc vào giờ cuối năm 2025.")

samples, label = build_forecast_dataset(development, features, TARGET, HORIZON)
train = samples.loc[samples["label_time"] < VALIDATION_START].copy()
valid = samples.loc[samples["label_time"] >= VALIDATION_START].copy()
if train.empty or valid.empty or valid["label_time"].iloc[0] != VALIDATION_START:
    raise ValueError("Dữ liệu phải có mẫu train trước 2025 và đầy đủ giờ validation 2025.")

X_train, y_train = train[features].astype(np.float32), train[label]
X_valid, y_valid = valid[features].astype(np.float32), valid[label]
train_times, valid_times = train["label_time"], valid["label_time"]

display(pd.DataFrame.from_dict(
    {"Train": describe_split(train), "Validation": describe_split(valid)}, orient="index"
))
display(pd.DataFrame([
    {"Dữ liệu": "Train", "Số dòng": len(train), "Số feature": len(features)},
    {"Dữ liệu": "Validation", "Số dòng": len(valid), "Số feature": len(features)},
]))

## 3. Huấn luyện Random Forest và chọn cấu hình

Random Forest có các siêu tham số quan trọng:

- `n_estimators`: số cây trong rừng. Notebook cố định 150 cây cho mỗi cấu hình để cân bằng độ ổn định và thời gian chạy.
- `max_depth`: giới hạn độ sâu mỗi cây, giúp kiểm soát độ phức tạp.
- `min_samples_leaf`: số mẫu tối thiểu ở một lá; tăng giá trị này giúp dự báo bớt nhạy với nhiễu.
- `max_features`: tỷ lệ feature được xét khi tìm phép chia tốt nhất ở mỗi nút.

Thử 4 cấu hình cho mỗi cách biểu diễn nhãn (tổng cộng 8 lượt fit). Chọn theo **MAE Validation thấp nhất**, sau đó RMSE và độ sâu trung bình của các cây nếu đồng hạng. Tập Test không tham gia lựa chọn.

In [ ]:
configurations = list(ParameterGrid({
    "max_depth": [16, None],
    "min_samples_leaf": [5, 20],
    "max_features": [0.8],
}))
training_targets = {
    "Trực tiếp": y_train,
    "Biến thiên": y_train - X_train[TARGET],
}

configuration_table = pd.DataFrame(configurations)
configuration_table.insert(0, "Số cây", N_ESTIMATORS)
display(configuration_table)
print(f"Sẽ huấn luyện {len(configurations) * len(training_targets)} cấu hình trên Validation.")

In [ ]:
tuning_rows = []
models = {}
best_key = (float("inf"), float("inf"), float("inf"))
tuning_start = perf_counter()

for target_mode, training_target in training_targets.items():
    for config_index, settings in enumerate(configurations, start=1):
        model_name = f"{target_mode} — cấu hình {config_index}"
        candidate = RandomForestRegressor(
            n_estimators=N_ESTIMATORS,
            criterion="squared_error",
            bootstrap=True,
            max_samples=0.9,
            n_jobs=N_JOBS,
            random_state=SEED,
            **settings,
        )
        fit_start = perf_counter()
        candidate.fit(X_train, training_target)
        fit_seconds = perf_counter() - fit_start
        validation_prediction = predict_temperature(candidate, X_valid, target_mode)
        metrics = score(y_valid, validation_prediction)
        mean_depth = float(np.mean([tree.tree_.max_depth for tree in candidate.estimators_]))
        row = {
            "Nhãn học": target_mode,
            "Cấu hình": config_index,
            "Số cây": N_ESTIMATORS,
            **settings,
            "Độ sâu TB": mean_depth,
            "Thời gian fit (giây)": fit_seconds,
            **metrics,
        }
        tuning_rows.append(row)
        models[(target_mode, config_index)] = candidate
        rank = (metrics["MAE (°C)"], metrics["RMSE (°C)"], mean_depth)
        if rank < best_key:
            best_key = rank
            best_model = candidate
            best_target_mode = target_mode
            best_config_index = config_index
            best_settings = settings.copy()
            best_valid_prediction = validation_prediction.copy()
            best_valid_metrics = metrics.copy()
        print(f"{model_name}: MAE={metrics['MAE (°C)']:.5f} | "
              f"RMSE={metrics['RMSE (°C)']:.5f} | fit={fit_seconds:.1f}s")

tuning_seconds = perf_counter() - tuning_start
ranking = pd.DataFrame(tuning_rows).sort_values(
    ["MAE (°C)", "RMSE (°C)", "Độ sâu TB"]
).reset_index(drop=True)
print(f"\nHoàn tất {len(ranking)} cấu hình trong {tuning_seconds / 60:.1f} phút.")
display(ranking.round(5))
print(f"Cấu hình được chọn: nhãn học {best_target_mode.lower()}, {best_settings}.")

In [ ]:
best_by_mode = ranking.drop_duplicates("Nhãn học").set_index("Nhãn học")
metric_columns = ["MAE (°C)", "RMSE (°C)", "R²"]
validation_scores = pd.concat([
    best_by_mode[metric_columns],
    pd.DataFrame([score(y_valid, X_valid[TARGET])], index=["Persistence"]),
])
display(validation_scores.round(5))

fig, ax = plt.subplots(figsize=(9, 4), layout="constrained")
validation_scores["MAE (°C)"].sort_values().plot.bar(ax=ax, color="#08788a")
ax.set(title="MAE trên Validation 2025", ylabel="MAE (°C)", xlabel="Mô hình")
ax.grid(axis="y", alpha=0.2)
plt.show()
plt.close(fig)

## 4. Đánh giá trên dữ liệu 2026

Sau khi chốt cấu hình bằng Validation, giữ nguyên mô hình đã fit trên dữ liệu trước 2025. Đọc CSV test và ghép thêm đúng một giờ cuối năm 2025 làm ngữ cảnh để dự báo nhãn 00:00 ngày 01/01/2026. Điều này giúp giữ đủ các giờ trong CSV test. Trong backtest, quan sát đã có tại giờ $t$ được dùng để dự báo $t+1$, nhưng tham số mô hình không được cập nhật.

In [ ]:
external = read_processed(TEST_PATH, "external_processed_sha256")
if external["time"].iloc[0] != TEST_START:
    raise ValueError("Tập test phải bắt đầu lúc 00:00 ngày 01/01/2026.")

test_source = pd.concat([development.tail(HORIZON), external], ignore_index=True)
test, test_label = build_forecast_dataset(test_source, features, TARGET, HORIZON)
if len(test) != len(external) or not test["label_time"].equals(external["time"].rename("label_time")):
    raise ValueError("Các nhãn dự báo phải phủ đúng các giờ trong CSV test.")

X_test, y_test = test[features].astype(np.float32), test[test_label]
pred_train = predict_temperature(best_model, X_train, best_target_mode)
pred_valid = predict_temperature(best_model, X_valid, best_target_mode)
pred_test = predict_temperature(best_model, X_test, best_target_mode)

predictions = test[["time", "label_time", test_label]].rename(columns={test_label: "Thực tế"})
predictions["Random Forest"] = pred_test
predictions["Persistence"] = test[TARGET].to_numpy()

result_rows = []
for split_name, actual, prediction, persistence in [
    ("Train", y_train, pred_train, X_train[TARGET].to_numpy()),
    ("Validation", y_valid, pred_valid, X_valid[TARGET].to_numpy()),
    ("Test", y_test, pred_test, X_test[TARGET].to_numpy()),
]:
    for model_name, values in [("Random Forest", prediction), ("Persistence", persistence)]:
        result_rows.append({"Tập": split_name, "Mô hình": model_name,
                            "Số mẫu": len(actual), **score(actual, values)})
results = pd.DataFrame(result_rows)
test_scores = results.loc[results["Tập"] == "Test"].set_index("Mô hình")

display(pd.DataFrame([describe_split(test)], index=["Test 2026"]))
display(results.round(5))
display(predictions.head(10).round(3))

## 5. Phân tích sai số và trực quan hóa dự báo

Biểu đồ chuỗi thời gian hiển thị tuần đầu tiên của test để đối chiếu dự báo với nhiệt độ thực tế. MAE theo tháng giúp kiểm tra mô hình có giai đoạn nào dự báo kém hơn rõ rệt hay không. Persistence là mốc đơn giản: dự báo nhiệt độ giờ tiếp theo bằng nhiệt độ tại giờ hiện tại.

In [ ]:
monthly = predictions.assign(
    month=predictions["label_time"].dt.to_period("M").astype(str),
    rf_error=(predictions["Thực tế"] - predictions["Random Forest"]).abs(),
    persistence_error=(predictions["Thực tế"] - predictions["Persistence"]).abs(),
).groupby("month").agg(
    samples=("Thực tế", "size"),
    random_forest_mae=("rf_error", "mean"),
    persistence_mae=("persistence_error", "mean"),
)
display(monthly.rename(columns={
    "samples": "Số giờ", "random_forest_mae": "MAE Random Forest (°C)",
    "persistence_mae": "MAE Persistence (°C)",
}).round(4))

week = predictions.head(min(7 * 24, len(predictions)))
fig, axes = plt.subplots(2, 1, figsize=(13, 7), layout="constrained")
axes[0].plot(week["label_time"], week["Thực tế"], label="Thực tế", color="#20252b", linewidth=1.5)
axes[0].plot(week["label_time"], week["Random Forest"], label="Random Forest", color="#08788a", linewidth=1.2)
axes[0].plot(week["label_time"], week["Persistence"], label="Persistence", color="#b06a27", linestyle="--")
axes[0].set(title="Actual vs Predicted — tuần đầu Test 2026", ylabel="Nhiệt độ (°C)")
axes[0].legend()
axes[1].plot(monthly.index, monthly["random_forest_mae"], "o-", label="Random Forest", color="#08788a")
axes[1].plot(monthly.index, monthly["persistence_mae"], "s--", label="Persistence", color="#b06a27")
axes[1].set(title="MAE theo tháng trên Test 2026", xlabel="Tháng", ylabel="MAE (°C)")
axes[1].legend()
for ax in axes:
    ax.grid(alpha=0.2)
plt.show()
plt.close(fig)

residuals = predictions["Thực tế"] - predictions["Random Forest"]
fig, ax = plt.subplots(figsize=(9, 4), layout="constrained")
ax.hist(residuals, bins=40, edgecolor="white")
ax.axvline(0, color="#b33a3a", linestyle="--", linewidth=1)
ax.set(title="Phân phối residual — Test 2026", xlabel="Residual (thực tế − dự báo), °C", ylabel="Số giờ")
ax.grid(axis="y", alpha=0.2)
plt.show()
plt.close(fig)

## 6. Feature Importance

`feature_importances_` là impurity-based importance: tổng mức giảm impurity được quy cho các lần chia sử dụng feature đó, sau khi chuẩn hóa thành tỷ lệ. Đây là chỉ báo nội bộ của mô hình, không phải quan hệ nhân quả; các feature tương quan có thể chia sẻ importance.

In [ ]:
importance = pd.DataFrame({
    "Feature": features,
    "Importance": best_model.feature_importances_,
})
importance["Importance (%)"] = importance["Importance"] * 100
importance = importance.sort_values("Importance", ascending=False).reset_index(drop=True)
display(importance[["Feature", "Importance (%)"]].head(15).round(5))

top_features = importance.head(15).sort_values("Importance (%)")
fig, ax = plt.subplots(figsize=(10, 6), layout="constrained")
ax.barh(top_features["Feature"], top_features["Importance (%)"], color="#08788a")
ax.set(title="Top 15 feature importance — Random Forest",
       xlabel="Importance (%)", ylabel="Feature")
ax.grid(axis="x", alpha=0.2)
plt.show()
plt.close(fig)

## 7. Tổng kết và lưu kết quả

Notebook lưu mô hình tốt nhất, bảng tuning, metrics và dự đoán có timestamp để có thể kiểm tra lại hoặc dùng ở bước kết hợp nhiều mô hình. Các trọng số ensemble (nếu có) không được tính trong notebook này.

In [ ]:
best_test = test_scores.loc["Random Forest"]
persistence_test = test_scores.loc["Persistence"]
mae_improvement = 100 * (1 - best_test["MAE (°C)"] / persistence_test["MAE (°C)"])

summary = pd.DataFrame([{
    "Cấu hình tốt nhất": best_target_mode,
    "Số cây": N_ESTIMATORS,
    "Siêu tham số": json.dumps(best_settings, ensure_ascii=False),
    "Feature": len(features),
    "Train": len(y_train),
    "Validation": len(y_valid),
    "Test": len(y_test),
    "Thời gian tuning (giây)": tuning_seconds,
    "Validation MAE (°C)": best_valid_metrics["MAE (°C)"],
    "Validation RMSE (°C)": best_valid_metrics["RMSE (°C)"],
    "Test MAE (°C)": best_test["MAE (°C)"],
    "Test RMSE (°C)": best_test["RMSE (°C)"],
    "Test R²": best_test["R²"],
    "MAE cải thiện so Persistence (%)": mae_improvement,
}])
display(summary.round(6))

# Lưu các artifact để dùng lại và phục vụ bước kết hợp mô hình.
joblib.dump(best_model, OUTPUT_DIR / "random_forest_model.joblib")
ranking.to_csv(OUTPUT_DIR / "random_forest_tuning_results.csv", index=False)
results.to_csv(OUTPUT_DIR / "random_forest_metrics.csv", index=False)
importance.to_csv(OUTPUT_DIR / "random_forest_feature_importance.csv", index=False)
predictions.to_csv(OUTPUT_DIR / "random_forest_test_predictions.csv", index=False)
valid_predictions = valid[["time", "label_time", label]].rename(columns={label: "Thực tế"}).copy()
valid_predictions["Random Forest"] = pred_valid
valid_predictions["Persistence"] = X_valid[TARGET].to_numpy()
valid_predictions.to_csv(OUTPUT_DIR / "random_forest_validation_predictions.csv", index=False)
(OUTPUT_DIR / "random_forest_config.json").write_text(json.dumps({
    "target": TARGET,
    "horizon_hours": HORIZON,
    "features": features,
    "n_estimators": N_ESTIMATORS,
    "random_state": SEED,
    "target_mode": best_target_mode,
    "best_parameters": best_settings,
    "validation_start": str(VALIDATION_START),
    "test_start": str(TEST_START),
}, ensure_ascii=False, indent=2), encoding="utf-8")

print(f"Đã lưu mô hình và kết quả tại: {OUTPUT_DIR.relative_to(ROOT)}")
print("Các file: model.joblib, tuning_results.csv, metrics.csv, feature_importance.csv, "
      "validation_predictions.csv, test_predictions.csv, config.json")
if mae_improvement > 0:
    conclusion = f"Random Forest giảm MAE {mae_improvement:.2f}% so với Persistence trên tập test."
else:
    conclusion = f"Random Forest có MAE cao hơn Persistence {abs(mae_improvement):.2f}% trên tập test."
display(Markdown(
    f"Trên **{len(y_test):,} giờ test**, Random Forest đạt MAE **{best_test['MAE (°C)']:.4f} °C**, "
    f"RMSE **{best_test['RMSE (°C)']:.4f} °C** và R² **{best_test['R²']:.5f}**. "
    f"Persistence đạt MAE **{persistence_test['MAE (°C)']:.4f} °C**. {conclusion}"
))

Kết quả là backtest dự báo trước một giờ trên dữ liệu thời tiết Hà Nội đã xử lý từ ERA5. Mô hình được lựa chọn bằng Validation 2025 và chỉ đánh giá một lần trên tập Test 2026. Tập test đã được sử dụng trong một số thử nghiệm trước của dự án, vì vậy cần mô tả đây là kết quả backtest trên tập test đã có trong quy trình nhóm, không phải đánh giá độc lập hoàn toàn.

**Tham khảo**

- [RandomForestRegressor — scikit-learn](https://scikit-learn.org/1.7/modules/generated/sklearn.ensemble.RandomForestRegressor.html)
- [Ensembles of randomized trees — scikit-learn User Guide](https://scikit-learn.org/1.7/modules/ensemble.html#forest)
- [Time-related feature engineering — scikit-learn](https://scikit-learn.org/1.7/auto_examples/applications/plot_time_series_lagged_features.html)